# ⚡ Módulo 13 - Notebook 03: Window Functions Distribuidas

## 🪟 Ranking, Acumulados y Promedios Móviles a Escala

**Libro:** Saliendo de lo Pandito  
**Módulo:** 13 - PySpark SQL Window DeltaLake  
**Duración estimada:** 75 minutos  
**Dificultad:** 🔴 Avanzado  
**Plataforma:** Databricks Free Edition

---

## 🎯 Objetivos de aprendizaje

Al finalizar este notebook serás capaz de:

✅ **Crear** Window Functions con Window.partitionBy()  
✅ **Calcular** rankings distribuidos (row_number, rank, dense_rank)  
✅ **Generar** acumulados (running totals)  
✅ **Aplicar** lag/lead para comparaciones temporales  
✅ **Construir** promedios móviles

---

## 📋 Pre-requisitos

* ✅ Notebooks 13_01 y 13_02 completados
* ✅ Conocimiento de Window Functions en SQL
* ✅ Familiaridad con particiones en Spark

---

## 📚 Contenido

1. Teoría de Window Functions
2. Window.partitionBy() y orderBy()
3. Funciones de Ranking
4. Funciones de Lag/Lead
5. Acumulados y Promedios Móviles
6. Caso Integrador: Análisis de Ventas con Windows

---

## 💡 Por qué importa

**Window Functions son esenciales para:**

* 📈 **Ranking:** Top N por categoría
* 📉 **Comparaciones:** Ventas vs mes anterior
* 📊 **Acumulados:** Ventas YTD (Year-To-Date)
* 📄 **Tendencias:** Promedios móviles

**Operaciones imposibles con GROUP BY**

In [0]:
import pandas as pd
import numpy as np
from pyspark.sql import functions as F
from pyspark.sql.window import Window

print("💾 CARGANDO DATOS REALES DESDE UNITY CATALOG")
print("="*70)

CATALOG = "pandito_ds"
SCHEMA = "default"

try:
    # Cargar tabla de ventas de Los Andes Market (SPARK DataFrame)
    df = spark.table(f"{CATALOG}.{SCHEMA}.ventas_mensuales_mendoza_h3")
    
    # Agregar columnas de fecha para análisis temporal
    df = df.withColumn("año", F.year("fecha")) \
           .withColumn("mes", F.month("fecha")) \
           .withColumn("trimestre", F.quarter("fecha"))
    
    # Registrar como vista temporal
    df.createOrReplaceTempView("ventas")
    
    print(f"\n✅ Datos reales cargados exitosamente")
    print(f"   📊 Registros: {df.count():,}")
    print(f"   🗃️ Particiones: {df.rdd.getNumPartitions()}")
    print(f"   📍 Ubicación: Mendoza, Argentina (Los Andes Market)")
    
    # Mostrar muestra ordenada por fecha
    print(f"\n📊 Muestra de datos (ordenados por fecha):")
    df.select("fecha", "sucursal_nombre", "zona", "ventas") \
      .orderBy("fecha") \
      .show(10, truncate=False)
    
    print(f"\n🎯 Este notebook aplicará Window Functions sobre datos REALES:")
    print(f"   • Ranking de sucursales por zona")
    print(f"   • Ventas acumuladas por mes")
    print(f"   • Comparación vs período anterior")
    print(f"   • Promedios móviles de 3 meses")
    
    USAR_DATOS_REALES = True
    
except Exception as e:
    print(f"\n⚠️  No se pudo cargar la tabla de Unity Catalog")
    print(f"   Error: {e}")
    print(f"\n📝 Solución:")
    print(f"   1. Ejecuta primero: 00_05_Preparacion_Datos_Empresariales.ipynb")
    print(f"   2. Verifica que la tabla exista: {CATALOG}.{SCHEMA}.ventas_mensuales_mendoza_h3")
    print(f"\n   Continuando con datos sintéticos...")
    
    df = None
    USAR_DATOS_REALES = False

print("\n" + "="*70)

## 📚 Window Functions: Operaciones Sobre Grupos

### 🪟 ¿Qué es una Window Function?

**Window Function:** Operación sobre un "grupo" de filas relacionadas **sin colapsar el resultado**.

**Diferencia con GROUP BY:**

**GROUP BY:** Colapsa filas
```sql
SELECT zona, SUM(ventas)
FROM ventas
GROUP BY zona
-- Resultado: 1 fila por zona
```

**WINDOW:** Mantiene todas las filas
```sql
SELECT 
    zona,
    ventas,
    SUM(ventas) OVER (PARTITION BY zona) as total_zona
FROM ventas
-- Resultado: TODAS las filas + total_zona
```

---

### 🛠️ Componentes de una Ventana

**Sintaxis:**
```python
from pyspark.sql.window import Window

window_spec = Window \
    .partitionBy("columna_grupo") \
    .orderBy("columna_orden")

df.withColumn("nueva_col", funcion().over(window_spec))
```

**Componentes:**

1. **PARTITION BY:** Divide en grupos (como GROUP BY)
2. **ORDER BY:** Orden dentro del grupo
3. **Función:** Qué calcular (rank, sum, lag, etc.)

---

### 🏆 Funciones de Ranking

**1️⃣ row_number():** Número secuencial (1, 2, 3...)
```python
F.row_number().over(Window.partitionBy("zona").orderBy(F.desc("ventas")))
# Zona Centro: 1, 2, 3, 4...
# Zona Norte:  1, 2, 3, 4...
```

**2️⃣ rank():** Ranking con gaps (1, 2, 2, 4...)
```python
F.rank().over(Window.partitionBy("zona").orderBy(F.desc("ventas")))
# Si hay empate en 2do lugar: 1, 2, 2, 4
```

**3️⃣ dense_rank():** Ranking sin gaps (1, 2, 2, 3...)
```python
F.dense_rank().over(Window.partitionBy("zona").orderBy(F.desc("ventas")))
# Si hay empate: 1, 2, 2, 3
```

---

### 🔙 Funciones Lag/Lead

**Acceder a filas anteriores/siguientes:**

**lag():** Fila anterior
```python
window = Window.partitionBy("sucursal").orderBy("fecha")

df.withColumn(
    "ventas_mes_anterior",
    F.lag("ventas", 1).over(window)  # 1 = 1 fila atrás
)
```

**lead():** Fila siguiente
```python
df.withColumn(
    "ventas_mes_siguiente",
    F.lead("ventas", 1).over(window)
)
```

**Caso de uso:** Calcular variación vs período anterior
```python
df.withColumn("variacion", F.col("ventas") - F.col("ventas_mes_anterior"))
```

---

### 📈 Acumulados (Running Totals)

**Suma acumulada:**
```python
window = Window.partitionBy("sucursal").orderBy("fecha")

df.withColumn(
    "ventas_acumuladas",
    F.sum("ventas").over(window)
)
```

**Resultado:**
```
Sucursal  Fecha       Ventas  Acumulado
A         2024-01     100     100
A         2024-02     150     250
A         2024-03     120     370
```

---

### 📉 Promedios Móviles

**Promedio de las últimas N filas:**

```python
window = Window \
    .partitionBy("sucursal") \
    .orderBy("fecha") \
    .rowsBetween(-2, 0)  # Fila actual + 2 anteriores

df.withColumn(
    "promedio_movil_3m",
    F.avg("ventas").over(window)
)
```

**Uso:** Suavizar tendencias y detectar patrones.

---

### 🔢 Frame Specification

**rowsBetween():** Controla qué filas incluir.

```python
# Últimas 3 filas (incluyendo actual)
rowsBetween(-2, 0)

# Todas las filas hasta la actual
rowsBetween(Window.unboundedPreceding, 0)

# Todas las filas de la partición
rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing)
```

---

### 💼 Caso de Uso: Ranking de Vendedores

```python
from pyspark.sql.window import Window
import pyspark.sql.functions as F

# Ventana: Por zona, ordenado por ventas desc
window = Window.partitionBy("zona").orderBy(F.desc("ventas"))

# Agregar ranking
df_ranked = df.withColumn("ranking", F.row_number().over(window))

# Filtrar top 3 por zona
top3 = df_ranked.filter(F.col("ranking") <= 3)
```

**Resultado:** Top 3 vendedores de cada zona.

In [0]:
import pandas as pd
import numpy as np
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import *
import warnings
warnings.filterwarnings('ignore')

print("🪟 WINDOW FUNCTIONS DISTRIBUIDAS")
print("="*70)

print(f"\nVersión de Pandas: {pd.__version__}")
print(f"Versión de NumPy: {np.__version__}")

try:
    print(f"Versión de Spark: {spark.version}")
except:
    print("⚠️  SparkSession no disponible")

print("\n🎯 En este notebook aprenderás:")
print("  • Window.partitionBy() y orderBy()")
print("  • row_number(), rank(), dense_rank()")
print("  • lag() y lead() - Comparaciones temporales")
print("  • sum().over(window) - Acumulados")
print("  • rowsBetween() - Promedios móviles")

print("\n📖 Métodos clave:")
print("  - Window.partitionBy('col').orderBy('col')")
print("  - F.row_number().over(window)")
print("  - F.lag('col', 1).over(window)")
print("  - F.sum('col').over(window)")
print("  - .rowsBetween(-2, 0)  # Promedio móvil 3")

print("\n" + "="*70)
print("✅ Librerías cargadas correctamente")

In [0]:
# Código de inicialización de notebook reindexado
import pandas as pd
import numpy as np
print('Notebook reindexado listo para práctica en Databricks')